# 04 — Diagnósticos de los Modelos MCO
**IA Generativa y Saber Pro 2021-2024** | Eduardo A. Victoria Cadena

Pruebas: Shapiro-Wilk/KS · Breusch-Pagan · VIF · Durbin-Watson · RESET de Ramsey · Corrección múltiple Holm/BH

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'
import sys; sys.path.insert(0, RUTA_PROYECTO + '/python')

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (het_breuschpagan, linear_reset)
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from utils import (MODULOS_GENERICOS, ETIQUETAS_VARS, CONTROLES, DUMMIES_DEPTO,
                   ALPHA, guardar_tabla, guardar_figura)

PROC_DIR = f'{RUTA_PROYECTO}/data/processed'
TBL_DIR  = f'{RUTA_PROYECTO}/outputs/tablas'
FIG_DIR  = f'{RUTA_PROYECTO}/outputs/figuras'
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_pickle(f'{PROC_DIR}/datos_analisis.pkl')
tabla4 = pd.read_csv(f'{TBL_DIR}/T4_coeficientes_todos_modelos.csv')
print("Datos y Tabla 4 cargados.")

In [ ]:
# ── Re-estimar modelos base para los diagnósticos ────────────────────────
def re_estimar_base(df, var_dep):
    rhs = ['periodo_ia']
    rhs += [d for d in DUMMIES_DEPTO if d in df.columns]
    if 'distancia_bogota_km' in df.columns: rhs.append('distancia_bogota_km')
    rhs += [c for c in CONTROLES if c in df.columns]
    formula = f'{var_dep} ~ ' + ' + '.join(rhs)
    cols    = [var_dep] + [c for c in rhs if c in df.columns and not c.startswith('C(')]
    df_c    = df[cols].dropna()
    if len(df_c) < 30: return None
    return smf.ols(formula, data=df_c).fit()

vars_dep = ['puntaje_saberpro_generico'] + [m for m in MODULOS_GENERICOS if m in df.columns]
modelos_base = {vd: re_estimar_base(df, vd) for vd in vars_dep}
print("Modelos base re-estimados:", list(modelos_base.keys()))

In [ ]:
# ── 1. Normalidad de residuales ───────────────────────────────────────────
def prueba_normalidad(mod, nombre):
    res = mod.resid.dropna()
    n   = len(res)
    if n < 5000:
        stat, p = stats.shapiro(res)
        prueba = 'Shapiro-Wilk'
    else:
        stat, p = stats.kstest(stats.zscore(res), 'norm')
        prueba = 'Kolmogorov-Smirnov'
    return {
        'modelo'      : nombre,
        'prueba'      : prueba,
        'estadistico' : round(stat, 4),
        'p_valor'     : round(p, 4),
        'n'           : n,
        'conclusion'  : 'No se rechaza normalidad' if p >= ALPHA else 'Se rechaza normalidad'
    }

rows_norm = [prueba_normalidad(mod, vd) for vd, mod in modelos_base.items() if mod]
tbl_norm  = pd.DataFrame(rows_norm)
print("=== Normalidad ===")
print(tbl_norm.to_string(index=False))

In [ ]:
# ── 2. Homocedasticidad: Breusch-Pagan ───────────────────────────────────
def prueba_bp(mod, nombre):
    bp_test = het_breuschpagan(mod.resid, mod.model.exog)
    stat, p, f_stat, f_p = bp_test
    return {
        'modelo'      : nombre,
        'BP_stat'     : round(stat, 4),
        'p_valor'     : round(p, 4),
        'conclusion'  : 'Homocedasticidad' if p >= ALPHA else 'Heterocedasticidad detectada'
    }

rows_bp = [prueba_bp(mod, vd) for vd, mod in modelos_base.items() if mod]
tbl_bp  = pd.DataFrame(rows_bp)
print("\n=== Breusch-Pagan ===")
print(tbl_bp.to_string(index=False))

In [ ]:
# ── 3. Multicolinealidad: VIF ─────────────────────────────────────────────
def prueba_vif(mod, nombre):
    X    = mod.model.exog
    cols = mod.model.exog_names
    rows = []
    for i, c in enumerate(cols):
        if c == 'Intercept': continue
        try:   v = variance_inflation_factor(X, i)
        except: v = np.nan
        rows.append({'modelo':nombre,'variable':c,'vif':round(v,3),
                     'alerta': v > 10 if not np.isnan(v) else False})
    return pd.DataFrame(rows)

tbl_vif_full = pd.concat([prueba_vif(mod, vd)
                           for vd, mod in modelos_base.items() if mod],
                           ignore_index=True)
max_vif = tbl_vif_full.groupby('modelo')['vif'].max().reset_index(name='max_vif')
print("\n=== VIF máximo por modelo ===")
print(max_vif.to_string(index=False))
alertas = tbl_vif_full[tbl_vif_full['alerta']]
print(f"\nVariables con VIF > 10: {len(alertas)}")
if len(alertas): print(alertas.to_string(index=False))

In [ ]:
# ── 4. Autocorrelación: Durbin-Watson ────────────────────────────────────
def prueba_dw(mod, nombre):
    dw = durbin_watson(mod.resid)
    return {
        'modelo'     : nombre,
        'DW_stat'    : round(dw, 4),
        'conclusion' : ('Posible autocorrelación positiva' if dw < 1.5
                        else 'Posible autocorrelación negativa' if dw > 2.5
                        else 'Sin evidencia de autocorrelación')
    }

tbl_dw = pd.DataFrame([prueba_dw(mod, vd)
                        for vd, mod in modelos_base.items() if mod])
print("\n=== Durbin-Watson ===")
print(tbl_dw.to_string(index=False))

In [ ]:
# ── 5. Especificación: RESET de Ramsey ───────────────────────────────────
def prueba_reset(mod, nombre):
    try:
        rst = linear_reset(mod, power=3, use_f=True)
        return {'modelo':nombre,'F_stat':round(rst.fvalue,4),
                'p_valor':round(rst.pvalue,4),
                'conclusion': 'Sin evidencia de mala especificación'
                              if rst.pvalue >= ALPHA else 'Posible mala especificación'}
    except Exception as e:
        return {'modelo':nombre,'F_stat':np.nan,'p_valor':np.nan,'conclusion':str(e)}

tbl_reset = pd.DataFrame([prueba_reset(mod, vd)
                           for vd, mod in modelos_base.items() if mod])
print("\n=== RESET de Ramsey ===")
print(tbl_reset.to_string(index=False))

In [ ]:
# ── 6. Tabla consolidada de diagnósticos ─────────────────────────────────
diag_resumen = (tbl_norm.rename(columns={'p_valor':'p_norm'})
                .merge(tbl_bp.rename(columns={'p_valor':'p_bp'})[['modelo','p_bp','conclusion']].rename(columns={'conclusion':'concl_bp'}), on='modelo')
                .merge(max_vif, on='modelo')
                .merge(tbl_dw, on='modelo')
                .merge(tbl_reset.rename(columns={'p_valor':'p_reset'})[['modelo','p_reset','conclusion']].rename(columns={'conclusion':'concl_reset'}), on='modelo'))

guardar_tabla(diag_resumen, 'T_diagnosticos_resumen', TBL_DIR)
print("\n=== Resumen de diagnósticos ===")
diag_resumen[['modelo','prueba','p_norm','p_bp','concl_bp','max_vif','DW_stat','p_reset']]

In [ ]:
# ── 7. Corrección por pruebas múltiples en β_IA ───────────────────────────
beta_ia_pvals = (tabla4[tabla4['variable']=='periodo_ia']
                 [['var_dep','spec','coef','se','p_valor']]
                 .dropna(subset=['p_valor'])
                 .copy())

_, p_holm, _, _ = multipletests(beta_ia_pvals['p_valor'], method='holm')
_, p_bh,   _, _ = multipletests(beta_ia_pvals['p_valor'], method='fdr_bh')
beta_ia_pvals['p_holm']    = p_holm.round(4)
beta_ia_pvals['p_bh']      = p_bh.round(4)
beta_ia_pvals['sig_orig']  = beta_ia_pvals['p_valor'] < ALPHA
beta_ia_pvals['sig_holm']  = p_holm < ALPHA
beta_ia_pvals['sig_bh']    = p_bh   < ALPHA

guardar_tabla(beta_ia_pvals, 'T_correccion_multiplas', TBL_DIR)
print("=== Corrección por pruebas múltiples (β_IA) ===")
print(beta_ia_pvals.to_string(index=False))

In [ ]:
# ── 8. Gráficos de diagnóstico del modelo genérico ───────────────────────
mod_gen = modelos_base.get('puntaje_saberpro_generico')
if mod_gen:
    res    = mod_gen.resid
    fitted = mod_gen.fittedvalues
    res_std= (res - res.mean()) / res.std()

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle('Diagnósticos del modelo — Puntaje Genérico (especificación base)',
                 fontsize=13, fontweight='bold')

    # Panel 1: Residuales vs Ajustados
    axes[0,0].scatter(fitted, res, alpha=0.3, s=6)
    axes[0,0].axhline(0, color='red', linestyle='--')
    axes[0,0].set(title='Residuales vs Ajustados',
                  xlabel='Valores ajustados', ylabel='Residuales')

    # Panel 2: QQ-plot
    sm.qqplot(res, line='s', ax=axes[0,1], alpha=0.4, markersize=3)
    axes[0,1].set_title('QQ-plot de residuales')

    # Panel 3: Escala-Localización
    axes[1,0].scatter(fitted, np.sqrt(np.abs(res_std)), alpha=0.3, s=6)
    axes[1,0].set(title='Escala-Localización',
                  xlabel='Valores ajustados',
                  ylabel='√|Residuales estandarizados|')

    # Panel 4: Histograma de residuales
    axes[1,1].hist(res, bins=40, density=True, alpha=0.6, color='steelblue')
    x = np.linspace(res.min(), res.max(), 300)
    axes[1,1].plot(x, stats.norm.pdf(x, res.mean(), res.std()),
                   'r--', linewidth=1.5, label='Normal teórica')
    axes[1,1].set(title='Distribución de residuales',
                  xlabel='Residual', ylabel='Densidad')
    axes[1,1].legend()

    plt.tight_layout()
    guardar_figura(fig, 'diag_puntaje_generico_base', FIG_DIR)
    plt.show()